In [2]:
import os
import cv2
import math
import random
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import matplotlib.pyplot as plt
import kagglehub

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# Download latest version
path = kagglehub.dataset_download("mariafrenti/age-prediction")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/mariafrenti/age-prediction


In [5]:
print(os.listdir(path)[:20])

['age_prediction_up', '20-50']


In [6]:
# for root, dirs, files in os.walk(path):
#     print("ROOT:", root)
#     print("DIRS:", dirs[:5])
#     print("FILES:", files[:5])
#     print("-" * 50)

In [7]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def crop_face_pil(image):
    img_np = np.array(image)
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    if len(faces) == 0:
        return image

    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])

    pad = int(0.25 * max(w, h))
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(img_np.shape[1], x + w + pad)
    y2 = min(img_np.shape[0], y + h + pad)

    cropped = img_np[y1:y2, x1:x2]
    return Image.fromarray(cropped)

In [8]:
def age_to_bin(age):
    if age <= 12:
        return 0
    elif age <= 19:
        return 1
    elif age <= 29:
        return 2
    elif age <= 39:
        return 3
    elif age <= 49:
        return 4
    elif age <= 59:
        return 5
    else:
        return 6

In [9]:
class AgeDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None, crop_face=False):
        self.transform = transform
        self.crop_face = crop_face
        self.samples = []
        valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

        for root, _, files in os.walk(root_dir):
            root_fixed = root.replace("\\", "/")

            if f"/{split}/" in root_fixed:
                folder_name = os.path.basename(root)

                if folder_name.isdigit():
                    age = float(folder_name)

                    for file in files:
                        if file.lower().endswith(valid_ext):
                            img_path = os.path.join(root, file)
                            self.samples.append((img_path, age))

        self.ages = [age for _, age in self.samples]
        self.min_age = min(self.ages)
        self.max_age = max(self.ages)

        age_counts = {}
        for age in self.ages:
            a = int(age)
            age_counts[a] = age_counts.get(a, 0) + 1

        max_count = max(age_counts.values())
        self.sample_weights = []
        for age in self.ages:
            a = int(age)
            self.sample_weights.append(max_count / age_counts[a])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, age = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.crop_face:
            image = crop_face_pil(image)

        if self.transform:
            image = self.transform(image)

        age_tensor = torch.tensor(age, dtype=torch.float32)
        age_bin = torch.tensor(age_to_bin(age), dtype=torch.long)

        return image, age_tensor, age_bin

In [10]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05)
    ], p=0.5),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = AgeDataset(path, split="train", transform=train_transform, crop_face=True)
val_dataset   = AgeDataset(path, split="test",  transform=test_transform,  crop_face=True)

print("Train size:", len(train_dataset))
print("Val size:",   len(val_dataset))
print("Age range:",  train_dataset.min_age, "to", train_dataset.max_age)

sample_weights = torch.tensor(train_dataset.sample_weights, dtype=torch.float32)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=48, sampler=sampler,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=48, shuffle=False,
                          num_workers=2, pin_memory=True)

In [ ]:
bin_counts = [0] * 7

for _, age in train_dataset.samples:
    bin = age_to_bin(age)
    bin_counts[bin] += 1

bin_counts = torch.tensor(bin_counts, dtype=torch.float32)

class_weights = bin_counts.sum() / (bin_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = class_weights.to(device)

print("Bin counts:", bin_counts)
print("Class weights:", class_weights)

In [ ]:
# best_model = AgeCNN().to(device)
# best_model.load_state_dict(torch.load("best_age_cnn.pth", map_location=device))
# best_model.eval()

In [ ]:
# for i in range(10):
#     image, true_age = val_dataset[i]
#     with torch.no_grad():
#         pred_age = best_model(image.unsqueeze(0).to(device)).item()
#     print(f"Sample {i+1} -> True age: {true_age.item():.0f}, Predicted age: {pred_age:.2f}")


# plt.figure(figsize=(15, 8))

In [ ]:
# for i in range(12):
#     image, true_age = val_dataset[i]

#     with torch.no_grad():
#         pred_age = best_model(image.unsqueeze(0).to(device)).item()

#     img = image.permute(1, 2, 0).cpu().numpy()
#     img = (img * 0.5) + 0.5
#     img = img.clip(0, 1)

#     plt.subplot(4, 6, i + 1)
#     plt.imshow(img)
#     plt.title(f"True: {true_age.item():.0f}\nPred: {pred_age:.1f}")
#     plt.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
# best_model = AgeCNN().to(device)
# best_model.load_state_dict(torch.load("best_age_cnn.pth", map_location=device))
# best_model.eval()

Pre-Trained

In [ ]:
from torchvision import models

In [ ]:
import timm

class AgeEfficientNetMultiTask(nn.Module):
    def __init__(self, min_age, max_age, num_bins=7):
        super().__init__()
        self.min_age = min_age
        self.max_age = max_age

        self.backbone = timm.create_model(
            "efficientnet_b4", pretrained=True, num_classes=0, global_pool="avg"
        )
        in_features = self.backbone.num_features  # 1792

        self.shared = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
        )

        self.age_head = nn.Linear(256, 1)

        self.bin_head = nn.Linear(256, num_bins)

    def forward(self, x, clamp_output=False):
        x = self.backbone(x)
        x = self.shared(x)

        age_out = self.age_head(x).squeeze(1)
        if clamp_output:
            age_out = age_out.clamp(self.min_age, self.max_age)

        bin_out = self.bin_head(x)
        return age_out, bin_out

In [ ]:
model = AgeEfficientNetMultiTask(train_dataset.min_age, train_dataset.max_age).to(device)

for param in model.backbone.parameters():
    param.requires_grad = False

In [ ]:
regression_criterion     = nn.HuberLoss(delta=5.0, reduction="mean")
classification_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3   
)

In [ ]:
train_mae_list = []
val_mae_list = []
within3_list = []

best_val_mae = float("inf")
epochs = 20

In [ ]:
for epoch in range(epochs):
    if epoch == 3:
        for block in list(model.backbone.blocks)[-2:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in model.backbone.conv_head.parameters():
            param.requires_grad = True
        for param in model.backbone.bn2.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW([
            {"params": filter(lambda p: p.requires_grad,
                              model.backbone.parameters()), "lr": 3e-5},
            {"params": model.shared.parameters(),           "lr": 3e-4},
            {"params": model.age_head.parameters(),         "lr": 3e-4},
            {"params": model.bin_head.parameters(),         "lr": 3e-4},
        ], weight_decay=1e-4)

    if epoch == 7:
        for param in model.backbone.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW([
            {"params": model.backbone.parameters(), "lr": 1e-5},
            {"params": model.shared.parameters(),   "lr": 1e-4},
            {"params": model.age_head.parameters(), "lr": 1e-4},
            {"params": model.bin_head.parameters(), "lr": 1e-4},
        ], weight_decay=1e-4)

    model.train()
    train_mae = 0
    train_total = 0
    print(f"Starting epoch {epoch+1}/{epochs}")

    for batch_idx, (images, ages, age_bins) in enumerate(train_loader):
        if batch_idx % 20 == 0:
            print(f"Epoch {epoch+1}, batch {batch_idx}/{len(train_loader)}")

        images   = images.to(device, non_blocking=True)
        ages     = ages.to(device, non_blocking=True)
        age_bins = age_bins.to(device, non_blocking=True)

        optimizer.zero_grad()
        pred_ages, pred_bins = model(images)

        reg_loss = regression_criterion(pred_ages, ages)
        cls_loss = classification_criterion(pred_bins, age_bins)
        loss     = reg_loss + 0.3 * cls_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_mae   += torch.sum(torch.abs(pred_ages - ages)).item()
        train_total += ages.size(0)

    train_mae /= train_total

    model.eval()
    val_mae  = 0
    total    = 0
    within_3 = 0
    within_5 = 0

    with torch.no_grad():
        for images, ages, age_bins in val_loader:
            images   = images.to(device, non_blocking=True)
            ages     = ages.to(device, non_blocking=True)
            age_bins = age_bins.to(device, non_blocking=True)

            pred_ages, pred_bins = model(images, clamp_output=True)

            val_mae  += torch.sum(torch.abs(pred_ages - ages)).item()
            diff      = torch.abs(pred_ages - ages)
            within_3 += (diff <= 3).sum().item()
            within_5 += (diff <= 5).sum().item()
            total    += ages.size(0)

    val_mae /= total
    acc3 = 100 * within_3 / total
    acc5 = 100 * within_5 / total

    scheduler.step(val_mae)

    train_mae_list.append(train_mae)
    val_mae_list.append(val_mae)
    within3_list.append(acc3)

    print(f"Epoch {epoch+1}/{epochs} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f} | Within ±3: {acc3:.2f}% | Within ±5: {acc5:.2f}%")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), "best_age_efficientnet_b4.pth")
        print("Best model saved!")

best_model = AgeEfficientNetMultiTask(train_dataset.min_age, train_dataset.max_age).to(device)
best_model.load_state_dict(torch.load("best_age_efficientnet_b4.pth", map_location=device))
best_model.eval()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_mae_list) + 1), train_mae_list, marker="o", label="Train MAE")
plt.plot(range(1, len(val_mae_list) + 1), val_mae_list, marker="o", label="Val MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("Train vs Val MAE")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(within3_list) + 1), within3_list, marker="o", label="Within ±3 Years")
plt.xlabel("Epoch")
plt.ylabel("Accuracy %")
plt.title("Validation Accuracy Within ±3 Years")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))

indices = random.sample(range(len(val_dataset)), 10)

for i, idx in enumerate(indices):
    image, true_age, true_bin = val_dataset[idx]

    with torch.no_grad():
        pred_age, pred_bin_logits = best_model(image.unsqueeze(0).to(device))
        pred_age = pred_age.item()
        pred_bin = torch.argmax(pred_bin_logits, dim=1).item()

    img = image.permute(1, 2, 0).cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)

    plt.subplot(2, 5, i + 1)
    plt.imshow(img)
    plt.title(f"True: {true_age.item():.0f}\nPred: {pred_age:.1f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
within_3 = 0
within_5 = 0
total = 0

best_model.eval()

with torch.no_grad():
    for images, ages, age_bins in val_loader:
        images = images.to(device)
        ages = ages.to(device)

        pred_ages, _ = best_model(images)
        diff = torch.abs(pred_ages - ages)

        within_3 += (diff <= 3).sum().item()
        within_5 += (diff <= 5).sum().item()
        total += ages.size(0)

print(f"Final Within ±3 Years: {100 * within_3 / total:.2f}%")
print(f"Final Within ±5 Years: {100 * within_5 / total:.2f}%")